# 03 — Multi-Agent Verified RAG

This notebook upgrades the baseline RAG system into a **claim-level fact-verification pipeline**.

```text
Question
   ↓
Retriever Agent
   ↓
Generator Agent
   ↓
Claim Extractor
   ↓
Verifier Agent (NLI)
   ↓
Guardrail
   ├── PASS → Final Answer
   └── FAIL → Repair → Re-verify
```

The same corpus, FAISS index, embedding model, and generator used in notebook 02 are retained where possible so that the comparison is meaningful.

**Important:** the verifier is an independent NLI model. It is not simply asking the generator whether its own answer is correct.


## 1. Install dependencies

In [1]:
!pip -q install transformers accelerate bitsandbytes sentence-transformers faiss-cpu pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.0 MB/s eta 0:00:00


In [2]:
import os
import shutil
from pathlib import Path

# 1. Clone your GitHub repository into /content/verified-rag
# (Note: Cell 2 in this notebook expects lowercase 'verified-rag')
REPO_URL = "https://github.com/faizzanasghar/Verified-RAG.git"
TARGET_DIR = "/content/verified-rag"

if not os.path.exists(TARGET_DIR):
    !git clone {REPO_URL} {TARGET_DIR}
else:
    print(f"{TARGET_DIR} already exists.")

# 2. Check if the vector store and data artifacts exist; if missing, generate them
required_files = [
    Path(TARGET_DIR) / "data/processed/corpus_metadata.json",
    Path(TARGET_DIR) / "data/processed/papers.json",
    Path(TARGET_DIR) / "data/vector_store/faiss.index",
    Path(TARGET_DIR) / "data/vector_store/chunks_metadata.json",
]

if any(not p.exists() for p in required_files):
    print("Corpus or FAISS index not found. Running 01_data_preparation.ipynb to build them...")
    %run {TARGET_DIR}/notebooks/01_data_preparation.ipynb
    print("Artifacts successfully generated.")
else:
    print("All required data artifacts and FAISS index are ready.")

Cloning into '/content/verified-rag'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 34 (delta 8), reused 26 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 934.87 KiB | 4.39 MiB/s, done.
Resolving deltas: 100% (8/8), done.
All required data artifacts and FAISS index are ready.


## 2. Imports and paths

In [3]:
from pathlib import Path
import json
import re
import time
from typing import Any, Dict

import numpy as np
import pandas as pd
import torch
import faiss

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)

PROJECT_ROOT = Path("/content/verified-rag")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
VECTOR_DIR = PROJECT_ROOT / "data" / "vector_store"
RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)


/content/verified-rag


## 3. Runtime check

In [4]:
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("CPU mode: inference will be slow.")


PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 14.56 GB


## 4. Load the fixed corpus and FAISS index

In [5]:
required = [
    PROCESSED_DIR / "corpus_metadata.json",
    PROCESSED_DIR / "papers.json",
    VECTOR_DIR / "faiss.index",
    VECTOR_DIR / "chunks_metadata.json",
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Run 01_data_preparation.ipynb first. Missing:\n" + "\n".join(missing)
    )

with (PROCESSED_DIR / "corpus_metadata.json").open(encoding="utf-8") as f:
    corpus_metadata = json.load(f)

with (PROCESSED_DIR / "papers.json").open(encoding="utf-8") as f:
    papers = json.load(f)

with (VECTOR_DIR / "chunks_metadata.json").open(encoding="utf-8") as f:
    chunks_metadata = json.load(f)

faiss_index = faiss.read_index(str(VECTOR_DIR / "faiss.index"))

print("Papers:", len(papers))
print("Chunks:", len(chunks_metadata))
print("Vectors:", faiss_index.ntotal)


Papers: 5
Chunks: 516
Vectors: 516


## 5. Retriever Agent

In [6]:
EMBEDDING_MODEL_NAME = corpus_metadata.get(
    "embedding_model",
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=embedding_device
)

def retrieve(query, top_k=5, min_score=None):
    vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(vector, top_k)
    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        score = float(score)
        if min_score is not None and score < min_score:
            continue
        item = chunks_metadata[int(idx)].copy()
        item["score"] = score
        results.append(item)

    return results

print("Embedding model:", EMBEDDING_MODEL_NAME)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2


## 6. Generator Agent

The generator is kept the same as notebook 02.

In [7]:
GENERATOR_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

generator_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL_NAME,
    trust_remote_code=True
)

if torch.cuda.is_available():
    q_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    generator_model = AutoModelForCausalLM.from_pretrained(
        GENERATOR_MODEL_NAME,
        quantization_config=q_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
else:
    generator_model = AutoModelForCausalLM.from_pretrained(
        GENERATOR_MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

generator_model.eval()
print("Loaded:", GENERATOR_MODEL_NAME)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-3B-Instruct


## 7. Verifier Agent — independent NLI model

The verifier checks whether a claim is entailed, contradicted, or unsupported by evidence.

In [8]:
NLI_MODEL_NAME = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)

nli_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_model.to(nli_device)
nli_model.eval()

print("Loaded:", NLI_MODEL_NAME)
print("Labels:", nli_model.config.id2label)


config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loaded: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


## 8. Shared agent state

In [9]:
def initial_state(question):
    return {
        "question": question,
        "retrieved_chunks": [],
        "draft_answer": "",
        "claims": [],
        "verification": [],
        "verification_score": 0.0,
        "status": "START",
        "iteration": 0,
        "final_answer": "",
        "logs": [],
        "timings": {},
    }


## 9. Retriever Agent

In [10]:
def retriever_agent(state, top_k=5):
    start = time.perf_counter()
    state["retrieved_chunks"] = retrieve(
        state["question"], top_k=top_k
    )
    state["timings"]["retrieval"] = time.perf_counter() - start
    state["logs"].append(
        f"Retriever: {len(state['retrieved_chunks'])} chunks"
    )
    return state


## 10. Generator Agent

In [11]:
GENERATOR_SYSTEM = """You are a technical research assistant.

Answer using ONLY the supplied evidence.
Every factual claim must have a citation in exactly this form:
[SOURCE: chunk_id]

Do not invent facts or source IDs.
If the evidence is insufficient, say so.
"""

def evidence_context(chunks):
    return "\n\n".join(
        f"[SOURCE: {x['chunk_id']}]\n"
        f"Paper: {x['title']}\n"
        f"Page: {x['page']}\n"
        f"Evidence:\n{x['text']}"
        for x in chunks
    )

def generator_agent(state):
    start = time.perf_counter()

    messages = [
        {"role": "system", "content": GENERATOR_SYSTEM},
        {"role": "user", "content":
            f"Question:\n{state['question']}\n\n"
            f"Evidence:\n{evidence_context(state['retrieved_chunks'])}\n\n"
            "Write the answer with source citations."
        },
    ]

    prompt = generator_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = generator_tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=8192
    )

    if torch.cuda.is_available():
        inputs = {k: v.to(generator_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = generator_model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            pad_token_id=generator_tokenizer.eos_token_id
        )

    n_input = inputs["input_ids"].shape[1]
    generated = outputs[0][n_input:]

    state["draft_answer"] = generator_tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

    state["timings"]["generation"] = time.perf_counter() - start
    state["logs"].append("Generator: draft created")
    return state


## 11. Claim Extractor

The answer is decomposed into atomic factual claims before verification.

In [12]:
CLAIM_PROMPT = """Extract atomic factual claims from the answer.

Return ONLY a numbered list.
Each item must contain exactly one independently verifiable factual claim.
Do not include citations, explanations, opinions, or the question.

ANSWER:
{answer}
"""

def parse_claims(text):
    claims = []
    for line in text.splitlines():
        line = line.strip()
        m = re.match(r"^(?:\d+[\).\:\-]|\-|\*)\s*(.+)$", line)
        if m and m.group(1).strip():
            claims.append(m.group(1).strip())
    return claims

def claim_extractor_agent(state):
    start = time.perf_counter()

    messages = [
        {"role": "system", "content": "Extract atomic factual claims."},
        {"role": "user", "content": CLAIM_PROMPT.format(
            answer=state["draft_answer"]
        )},
    ]

    prompt = generator_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = generator_tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=4096
    )

    if torch.cuda.is_available():
        inputs = {k: v.to(generator_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = generator_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=generator_tokenizer.eos_token_id
        )

    n_input = inputs["input_ids"].shape[1]
    extracted = generator_tokenizer.decode(
        outputs[0][n_input:], skip_special_tokens=True
    ).strip()

    claims = parse_claims(extracted)

    if not claims and state["draft_answer"].strip():
        claims = [state["draft_answer"].strip()]

    state["claims"] = [
        {"claim_id": f"claim_{i+1}", "text": x}
        for i, x in enumerate(claims)
    ]

    state["timings"]["claim_extraction"] = time.perf_counter() - start
    state["logs"].append(f"Claim Extractor: {len(claims)} claims")
    return state


## 12. Claim-specific retrieval

In [13]:
def retrieve_for_claim(claim, top_k=3):
    return retrieve(claim, top_k=top_k)


## 13. NLI scoring

In [14]:
def nli_predict(claim, evidence):
    inputs = nli_tokenizer(
        evidence,
        claim,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(nli_device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = nli_model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]

    scores = {}
    for i, p in enumerate(probs):
        label = nli_model.config.id2label[i].lower()
        if "entail" in label:
            scores["entailment"] = float(p)
        elif "contrad" in label:
            scores["contradiction"] = float(p)
        elif "neutral" in label:
            scores["neutral"] = float(p)

    return scores

def aggregate_nli(pairs):
    if not pairs:
        return {
            "label": "NOT_ENOUGH_EVIDENCE",
            "entailment": 0.0,
            "contradiction": 0.0,
            "neutral": 1.0
        }

    ent = max(x["nli"]["entailment"] for x in pairs)
    con = max(x["nli"]["contradiction"] for x in pairs)
    neu = max(x["nli"]["neutral"] for x in pairs)

    if con >= 0.80 and con > ent:
        label = "REFUTED"
    elif ent >= 0.70:
        label = "SUPPORTED"
    else:
        label = "NOT_ENOUGH_EVIDENCE"

    return {
        "label": label,
        "entailment": ent,
        "contradiction": con,
        "neutral": neu
    }


## 14. Verifier Agent

In [15]:
def verifier_agent(state, claim_top_k=3):
    start = time.perf_counter()
    results = []

    for claim_item in state["claims"]:
        claim = claim_item["text"]
        evidence_chunks = retrieve_for_claim(
            claim, top_k=claim_top_k
        )

        pairs = []
        for evidence in evidence_chunks:
            scores = nli_predict(
                claim,
                evidence["text"]
            )
            pairs.append({
                "chunk_id": evidence["chunk_id"],
                "paper_id": evidence["paper_id"],
                "title": evidence["title"],
                "page": evidence["page"],
                "retrieval_score": evidence["score"],
                "text": evidence["text"],
                "nli": scores,
            })

        aggregate = aggregate_nli(pairs)

        supporting = [
            x["chunk_id"] for x in pairs
            if x["nli"]["entailment"] >= 0.70
        ]

        results.append({
            "claim_id": claim_item["claim_id"],
            "claim": claim,
            "label": aggregate["label"],
            "entailment": aggregate["entailment"],
            "contradiction": aggregate["contradiction"],
            "neutral": aggregate["neutral"],
            "supporting_sources": supporting,
            "evidence": pairs,
        })

    state["verification"] = results

    state["verification_score"] = (
        np.mean([
            x["label"] == "SUPPORTED"
            for x in results
        ]) if results else 0.0
    )

    state["timings"]["verification"] = time.perf_counter() - start
    state["logs"].append("Verifier: claim-level verification complete")
    return state


## 15. Guardrail

In [16]:
def guardrail(state):
    labels = [x["label"] for x in state["verification"]]

    if labels and all(x == "SUPPORTED" for x in labels):
        state["status"] = "PASS"
    else:
        state["status"] = "FAIL"

    state["logs"].append(
        f"Guardrail: {state['status']}"
    )
    return state


## 16. Repair Agent

Unsupported or refuted claims are removed from the answer and the model is asked to rewrite using only verified evidence.

In [17]:
REPAIR_SYSTEM = """You are a factuality-focused technical answer editor.

Rewrite the answer using ONLY claims labeled SUPPORTED.
Remove REFUTED and NOT_ENOUGH_EVIDENCE claims.
Do not introduce new facts.
Use only the verified source IDs.
Every factual claim must contain [SOURCE: chunk_id].
If insufficient evidence remains, say so.
Return only the final answer.
"""

def repair_agent(state):
    start = time.perf_counter()

    summary = "\n".join(
        f"{x['claim_id']}: {x['label']} — {x['claim']}"
        for x in state["verification"]
    )

    verified_evidence = []
    for x in state["verification"]:
        if x["label"] != "SUPPORTED":
            continue
        for e in x["evidence"]:
            if e["nli"]["entailment"] >= 0.70:
                verified_evidence.append(
                    f"[SOURCE: {e['chunk_id']}]\n{e['text']}"
                )

    prompt = f"""Question:
{state['question']}

Draft:
{state['draft_answer']}

Verification:
{summary}

Verified evidence:
{chr(10).join(verified_evidence)}

Rewrite using only supported claims.
"""

    messages = [
        {"role": "system", "content": REPAIR_SYSTEM},
        {"role": "user", "content": prompt},
    ]

    text = generator_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = generator_tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=8192
    )

    if torch.cuda.is_available():
        inputs = {k: v.to(generator_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = generator_model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            pad_token_id=generator_tokenizer.eos_token_id
        )

    n_input = inputs["input_ids"].shape[1]
    state["draft_answer"] = generator_tokenizer.decode(
        outputs[0][n_input:], skip_special_tokens=True
    ).strip()

    state["iteration"] += 1
    state["timings"]["repair"] = (
        state["timings"].get("repair", 0.0)
        + time.perf_counter() - start
    )
    state["logs"].append(
        f"Repair Agent: iteration {state['iteration']}"
    )
    return state


## 17. Complete multi-agent workflow

In [18]:
def run_verified_rag(
    question,
    top_k=5,
    claim_top_k=3,
    max_repairs=2
):
    state = initial_state(question)

    state = retriever_agent(state, top_k)
    state = generator_agent(state)

    while True:
        state = claim_extractor_agent(state)
        state = verifier_agent(state, claim_top_k)
        state = guardrail(state)

        if state["status"] == "PASS":
            break

        if state["iteration"] >= max_repairs:
            break

        state = repair_agent(state)

    state["final_answer"] = state["draft_answer"]
    state["timings"]["total"] = sum(state["timings"].values())

    return state


## 18. Run one complete example

In [19]:
question = (
    "What problem does residual learning address, "
    "and how do shortcut connections help?"
)

state = run_verified_rag(
    question,
    top_k=5,
    claim_top_k=3,
    max_repairs=2
)

print("QUESTION\n", state["question"])
print("\nFINAL ANSWER\n", state["final_answer"])
print("\nSTATUS:", state["status"])
print("REPAIRS:", state["iteration"])
print("VERIFICATION SCORE:", round(float(state["verification_score"]), 3))

print("\nCLAIMS")
for x in state["verification"]:
    print("-" * 90)
    print(x["claim_id"], "|", x["label"])
    print(x["claim"])
    print(
        "entailment=", round(x["entailment"], 3),
        "contradiction=", round(x["contradiction"], 3),
        "neutral=", round(x["neutral"], 3)
    )
    print("sources:", x["supporting_sources"])

print("\nLOG")
for x in state["logs"]:
    print("-", x)


QUESTION
 What problem does residual learning address, and how do shortcut connections help?

FINAL ANSWER
 Residual learning addresses the problem of degradation (degradation problem) when training deep neural networks. Shortcut connections help by adding a direct path (shortcut) from earlier layers to later layers, which allows the network to learn residual functions. This means that the network can learn to add or subtract certain values without having to relearn them at each layer, thus mitigating the degradation problem and enabling the network to generalize better and achieve higher accuracy gains with increased depth.

[SOURCE: resnet_2015_p005_c004]
[SOURCE: resnet_2015_p002_c007]

STATUS: FAIL
REPAIRS: 2
VERIFICATION SCORE: 0.0

CLAIMS
------------------------------------------------------------------------------------------
claim_1 | NOT_ENOUGH_EVIDENCE
Residual learning addresses the problem of degradation (degradation problem) when training deep neural networks. Shortcut co

## 19. Inspect claim evidence

In [20]:
for claim in state["verification"]:
    print("=" * 110)
    print(claim["claim_id"], "|", claim["label"])
    print("CLAIM:", claim["claim"])

    for evidence in claim["evidence"]:
        print("-" * 100)
        print(
            evidence["chunk_id"],
            "| retrieval=", round(evidence["retrieval_score"], 4),
            "| entailment=", round(evidence["nli"]["entailment"], 3),
            "| contradiction=", round(evidence["nli"]["contradiction"], 3)
        )
        print(evidence["text"][:1200])


claim_1 | NOT_ENOUGH_EVIDENCE
CLAIM: Residual learning addresses the problem of degradation (degradation problem) when training deep neural networks. Shortcut connections help by adding a direct path (shortcut) from earlier layers to later layers, which allows the network to learn residual functions. This means that the network can learn to add or subtract certain values without having to relearn them at each layer, thus mitigating the degradation problem and enabling the network to generalize better and achieve higher accuracy gains with increased depth.

[SOURCE: resnet_2015_p005_c004]
[SOURCE: resnet_2015_p002_c007]
----------------------------------------------------------------------------------------------------
resnet_2015_p005_c004 | retrieval= 0.7563 | entailment= 0.055 | contradiction= 0.015
ason for such optimization difﬁculties will be studied in the future. Residual Networks. Next we evaluate 18-layer and 34layer residual nets (ResNets) The baseline architectures are the s

## 20. Benchmark questions

In [21]:
BENCHMARK_QUESTIONS = [
    {"id": "q01", "question": "What problem does residual learning address in very deep neural networks?"},
    {"id": "q02", "question": "What is the main architectural idea of the Transformer?"},
    {"id": "q03", "question": "Why does the Transformer not require recurrence to model relationships between tokens?"},
    {"id": "q04", "question": "What is the purpose of pre-training in BERT?"},
    {"id": "q05", "question": "What are the two moment estimates used by Adam?"},
    {"id": "q06", "question": "How does Adam adapt the learning rate for individual parameters?"},
]

display(pd.DataFrame(BENCHMARK_QUESTIONS))


,id,question
0,q01,What problem does residual learning address in...
1,q02,What is the main architectural idea of the Tra...
2,q03,Why does the Transformer not require recurrenc...
3,q04,What is the purpose of pre-training in BERT?
4,q05,What are the two moment estimates used by Adam?
5,q06,How does Adam adapt the learning rate for indi...


## 21. Run the verified-RAG benchmark

In [22]:
verified_outputs = []

for item in tqdm(BENCHMARK_QUESTIONS, desc="Verified RAG"):
    verified_outputs.append(
        run_verified_rag(
            item["question"],
            top_k=5,
            claim_top_k=3,
            max_repairs=2
        )
    )

print("Completed:", len(verified_outputs))


Verified RAG:   0%|          | 0/6 [00:00<?, ?it/s]

Completed: 6


## 22. Review outputs

In [23]:
for state in verified_outputs:
    print("=" * 110)
    print(state["question"])
    print("-" * 110)
    print(state["final_answer"])
    print("\nStatus:", state["status"])
    print("Repairs:", state["iteration"])
    print("Claims:", len(state["claims"]))
    print("Support rate:", round(float(state["verification_score"]), 3))
    print()


What problem does residual learning address in very deep neural networks?
--------------------------------------------------------------------------------------------------------------
[SOURCE: resnet_2015_p001_c000]
[SOURCE: resnet_2015_p002_c002]
[SOURCE: resnet_2015_p001_c003]
[SOURCE: resnet_2015_p002_c008]

Residual learning addresses the problem of vanishing/exploding gradients in very deep neural networks. These problems hinder the convergence of deep networks, leading to higher training error and eventually degradation in performance as the network depth increases. By explicitly reformulating layers as learning residual functions, residual networks mitigate these issues and allow for greater depth without significant degradation in performance.

Verified evidence: SUPPORTED

Status: FAIL
Repairs: 2
Claims: 1
Support rate: 0.0

What is the main architectural idea of the Transformer?
-------------------------------------------------------------------------------------------------

## 23. Claim-level metrics

In [24]:
rows = []

for item in verified_outputs:
    labels = [x["label"] for x in item["verification"]]
    total = len(labels)

    supported = labels.count("SUPPORTED")
    refuted = labels.count("REFUTED")
    insufficient = labels.count("NOT_ENOUGH_EVIDENCE")

    rows.append({
        "question": item["question"],
        "claims": total,
        "supported": supported,
        "refuted": refuted,
        "not_enough_evidence": insufficient,
        "support_rate": supported / total if total else 0,
        "refutation_rate": refuted / total if total else 0,
        "unsupported_rate": insufficient / total if total else 0,
        "strict_pass": item["status"] == "PASS",
        "repairs": item["iteration"],
        "total_time_seconds": item["timings"]["total"],
    })

verified_metrics = pd.DataFrame(rows)
display(verified_metrics)

print(
    "Mean claim support rate:",
    round(verified_metrics["support_rate"].mean() * 100, 2), "%"
)
print(
    "Strict answer pass rate:",
    round(verified_metrics["strict_pass"].mean() * 100, 2), "%"
)


,question,claims,supported,refuted,not_enough_evidence,support_rate,refutation_rate,unsupported_rate,strict_pass,repairs,total_time_seconds
0,What problem does residual learning address in...,1,0,0,1,0.0,0.0,1.0,False,2,42.881176
1,What is the main architectural idea of the Tra...,1,0,0,1,0.0,0.0,1.0,False,2,68.712307
2,Why does the Transformer not require recurrenc...,1,0,1,0,0.0,1.0,0.0,False,2,50.449501
3,What is the purpose of pre-training in BERT?,1,0,0,1,0.0,0.0,1.0,False,2,68.249590
4,What are the two moment estimates used by Adam?,1,1,0,0,1.0,0.0,0.0,True,0,15.487037
5,How does Adam adapt the learning rate for indi...,1,0,0,1,0.0,0.0,1.0,False,2,60.719435


Mean claim support rate: 16.67 %
Strict answer pass rate: 16.67 %


## 24. Citation validity

In [25]:
SOURCE_PATTERN = re.compile(
    r"\[SOURCE:\s*([^\]]+)\]",
    flags=re.IGNORECASE
)

def citation_check(state):
    cited = [
        x.strip()
        for x in SOURCE_PATTERN.findall(state["final_answer"])
    ]
    valid_ids = {
        x["chunk_id"] for x in state["retrieved_chunks"]
    }

    return {
        "citations": cited,
        "valid": [x for x in cited if x in valid_ids],
        "invalid": [x for x in cited if x not in valid_ids],
    }

citation_rows = []

for state in verified_outputs:
    check = citation_check(state)
    citation_rows.append({
        "question": state["question"],
        "citations": len(check["citations"]),
        "valid": len(check["valid"]),
        "invalid": len(check["invalid"]),
        "invalid_rate": (
            len(check["invalid"]) / len(check["citations"])
            if check["citations"] else 0
        )
    })

citation_df = pd.DataFrame(citation_rows)
display(citation_df)


,question,citations,valid,invalid,invalid_rate
0,What problem does residual learning address in...,4,4,0,0.0
1,What is the main architectural idea of the Tra...,5,5,0,0.0
2,Why does the Transformer not require recurrenc...,1,1,0,0.0
3,What is the purpose of pre-training in BERT?,11,11,0,0.0
4,What are the two moment estimates used by Adam?,5,5,0,0.0
5,How does Adam adapt the learning rate for indi...,0,0,0,0.0


## 25. Save the complete experiment

In [26]:
experiment_name = "verified_rag_v1"

def json_safe(x):
    if isinstance(x, dict):
        return {k: json_safe(v) for k, v in x.items()}
    if isinstance(x, list):
        return [json_safe(v) for v in x]
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        return float(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    return x

metadata = {
    "experiment": experiment_name,
    "generator_model": GENERATOR_MODEL_NAME,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "nli_model": NLI_MODEL_NAME,
    "top_k": 5,
    "claim_top_k": 3,
    "max_repairs": 2,
    "entailment_threshold": 0.70,
    "contradiction_threshold": 0.80,
    "num_questions": len(verified_outputs),
    "runtime": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else "CPU"
    ),
}

payload = {
    "metadata": metadata,
    "results": json_safe(verified_outputs)
}

json_path = RESULTS_DIR / f"{experiment_name}.json"
metrics_path = RESULTS_DIR / f"{experiment_name}_metrics.csv"

with json_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

verified_metrics.to_csv(metrics_path, index=False)

print("Saved:", json_path)
print("Saved:", metrics_path)


Saved: /content/verified-rag/results/verified_rag_v1.json
Saved: /content/verified-rag/results/verified_rag_v1_metrics.csv


# Multi-Agent Verified RAG complete

The project now contains a genuine verification workflow rather than only ordinary RAG:

```text
Retriever → Generator → Claim Extraction
                     ↓
               Claim Retrieval
                     ↓
                  NLI
                     ↓
                 Guardrail
                ↙         ↘
             PASS         FAIL
              ↓             ↓
             END          Repair
                            ↓
                         Re-check
```

### Research artifacts produced

- claim-level verification records
- evidence for every claim
- supported/refuted/unsupported labels
- strict pass/fail decisions
- repair iterations
- latency measurements
- complete JSON experiment logs

`04_evaluation.ipynb` should compare the baseline and verified systems using retrieval, factuality, citation, latency, and repair metrics.

**Do not treat the small six-question benchmark as final evidence.** It is only a sanity check before building the proper evaluation benchmark.


In [ ]:
import json
from pathlib import Path
from google.colab import _message

# 1. Save current notebook JSON into the repo's notebooks/ folder
notebook_json = _message.blocking_request("get_ipynb")
target_nb_path = Path("/content/verified-rag/notebooks/03_multi_agent_rag.ipynb")

with open(target_nb_path, "w", encoding="utf-8") as f:
    json.dump(notebook_json["ipynb"], f, indent=2)

print(f"Saved notebook to {target_nb_path}")